# CWRU bearing signals: healthy vs inner-race fault

We compare `97.mat` (normal, 0 HP) and `105.mat` (12 kHz drive-end, 0.007 inch inner-race fault, 0 HP). CWRU lists both operating points at about 1797 rpm. Its official 12 kHz table identifies `105.mat`; the normal-baseline table lists `97.mat` and its speed but not a per-file sample rate. The paired spectrum check below estimates `97.mat`'s rate from its shaft-harmonic positions relative to `105.mat`.

Sources: [CWRU normal baseline](https://engineering.case.edu/bearingdatacenter/normal-baseline-data), [CWRU 12 kHz drive-end data](https://engineering.case.edu/bearingdatacenter/12k-drive-end-bearing-fault-data), [CWRU download notes](https://engineering.case.edu/bearingdatacenter/download-data-file).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat
from scipy.signal import find_peaks

from motorsense_ml.envelope import envelope, envelope_spectrum
from motorsense_ml.features import crest_factor, kurtosis, peak, rms
from motorsense_ml.resampling import resample_signal

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ml" / "data").is_dir())
DATA_DIR = ROOT / "ml" / "data"
FILES = {"healthy": DATA_DIR / "97.mat", "inner-race fault": DATA_DIR / "105.mat"}
WINDOW_SAMPLES = 4096

signals = {}
for label, path in FILES.items():
    mat = loadmat(path)
    keys = [key for key in mat if key.endswith("_DE_time")]
    rpm_keys = [key for key in mat if key.endswith("RPM")]
    if len(keys) != 1 or len(rpm_keys) != 1:
        raise ValueError(
            f"Expected one *_DE_time and one *RPM variable in {path.name}; "
            f"found {keys} and {rpm_keys}"
        )
    signal = np.asarray(mat[keys[0]]).reshape(-1)
    rpm = float(np.asarray(mat[rpm_keys[0]]).squeeze())
    signals[label] = (keys[0], signal, rpm)
    print(f"{label}: {keys[0]}, {signal.size:,} samples, {rpm:g} rpm")

In [ ]:
fig, axes = plt.subplots(len(signals), 1, figsize=(12, 6), sharex=True, constrained_layout=True)
for ax, (label, (_, signal, _)) in zip(axes, signals.items(), strict=False):
    shown = signal[:WINDOW_SAMPLES]
    ax.plot(np.arange(shown.size), shown, linewidth=0.7)
    ax.set_title(label)
    ax.set_ylabel("Acceleration (recorded units)")
axes[-1].set_xlabel("Sample index")
plt.show()

In [ ]:
feature_rows = []
for label, (_, signal, _) in signals.items():
    window = signal[:WINDOW_SAMPLES]
    feature_rows.append(
        {
            "condition": label,
            "rms": rms(window),
            "peak": peak(window),
            "crest_factor": crest_factor(window),
            "kurtosis": kurtosis(window),
        }
    )

for row in feature_rows:
    print(row)

## Check the sample rate using shaft speed

The normal recording's shaft speed is about 1797 rpm, or 29.95 Hz. Since the sampling rate is what we're checking, first view the spectrum in cycles per sample. If the selected peak is the `h`th shaft harmonic, then `fs = h × 29.95 / f_peak`. Compare the line and its harmonics against the 12 kHz and 48 kHz candidate locations. The strongest line alone is ambiguous: at 0.0075 cycles/sample it can be 3× at 12 kHz or 12× at 48 kHz. The paired comparison below checks several shaft harmonics against the known 12 kHz recording.

In [ ]:
_, normal, shaft_rpm = signals["healthy"]
x = normal[: 2**17]
x = x - x.mean()
N = len(x)
window = np.hanning(N)
xw = x * window
spectrum = np.abs(np.fft.rfft(xw))
freq_cycles_per_sample = np.fft.rfftfreq(N, d=1.0)

shaft_hz = shaft_rpm / 60
search = (freq_cycles_per_sample >= 2 / N) & (freq_cycles_per_sample <= 0.01)
peak_bin = np.flatnonzero(search)[np.argmax(spectrum[search])]
f_peak = freq_cycles_per_sample[peak_bin]
print(f"Largest low-frequency peak: bin {peak_bin}, {f_peak:.6f} cycles/sample")
for order in (1, 2, 3):
    print(f"If this is the {order}× shaft harmonic: fs ≈ {order * shaft_hz / f_peak:.0f} Hz")
for candidate_fs in (12_000, 48_000):
    peak_order = f_peak * candidate_fs / shaft_hz
    print(f"At fs={candidate_fs} Hz, selected peak is {peak_order:.1f}× shaft speed")
    for harmonic in (1, 2, 3):
        expected = harmonic * shaft_hz / candidate_fs
        center_bin = round(expected * N)
        lo = max(2, center_bin - 5)
        hi = min(len(spectrum), center_bin + 6)
        local_bin = lo + np.argmax(spectrum[lo:hi])
        print(
            f"  {harmonic}× expected {expected:.6f}: local peak "
            f"{freq_cycles_per_sample[local_bin]:.6f} cycles/sample"
        )

fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
ax.semilogy(freq_cycles_per_sample[2:], spectrum[2:], linewidth=0.8)
ax.axvline(f_peak, color="black", linestyle="-", label="selected peak")
for candidate_fs, color in ((12_000, "tab:blue"), (48_000, "tab:orange")):
    for harmonic in (1, 2, 3):
        ax.axvline(
            harmonic * shaft_hz / candidate_fs,
            color=color,
            linestyle="--",
            alpha=0.65,
            label=f"{harmonic}× shaft if fs={candidate_fs // 1000} kHz" if harmonic == 1 else None,
        )
ax.set_xlim(0, 0.01)
ax.set_xlabel("Frequency (cycles/sample)")
ax.set_ylabel("Magnitude (windowed FFT)")
ax.set_title("97.mat normal drive-end spectrum")
ax.legend()
plt.show()

## Compare 97.mat against the known 12 kHz recording

`105.mat` is listed by CWRU under 12 kHz drive-end data and both recordings run at nearly 1797 rpm. Their shaft fundamental should therefore have the same cycles/sample position if their rates match. If `97.mat` is 48 kHz, its 1× line should be at one quarter of `105.mat`'s 1× position. Detect peaks independently, then compare the predicted harmonic orders; do not select a maximum inside a band centered on the expected frequency.


In [ ]:
FFT_SAMPLES = 2**16
spectra = {}
for label, (_, signal, rpm) in signals.items():
    x = signal[:FFT_SAMPLES].astype(float)
    x = (x - x.mean()) * np.hanning(FFT_SAMPLES)
    magnitude = np.abs(np.fft.rfft(x))
    cycles_per_sample = np.fft.rfftfreq(FFT_SAMPLES, d=1.0)
    normalized_db = 20 * np.log10(magnitude / magnitude.max())
    peak_bins, _ = find_peaks(normalized_db, prominence=6, distance=5)
    peak_bins = peak_bins[cycles_per_sample[peak_bins] <= 0.02]
    spectra[label] = (cycles_per_sample, normalized_db, peak_bins, rpm)

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
for label, (freq, magnitude_db, _, _) in spectra.items():
    ax.plot(freq[1:], magnitude_db[1:], label=label, linewidth=0.9)

normal_rpm = signals["healthy"][2]
fault_rpm = signals["inner-race fault"][2]
for order in (1, 4, 8, 12):
    ax.axvline(
        order * normal_rpm / 60 / 48_000,
        color="tab:red",
        linestyle=":" if order != 1 else "--",
        alpha=0.8,
        label="97.mat shaft orders if fs=48 kHz" if order == 1 else None,
    )
for order in (1, 2, 3):
    ax.axvline(
        order * fault_rpm / 60 / 12_000,
        color="black",
        linestyle="--",
        alpha=0.65,
        label="105.mat shaft harmonics (12 kHz)" if order == 1 else None,
    )

ax.set_xlim(0, 0.02)
ax.set_xlabel("Frequency (cycles/sample)")
ax.set_ylabel("Magnitude (dB, normalized per recording)")
ax.set_title("Drive-end spectra: 97.mat vs 105.mat")
ax.grid(True, alpha=0.2)
ax.legend()
plt.show()

bin_width = 1 / FFT_SAMPLES
for label, (freq, magnitude_db, peak_bins, rpm) in spectra.items():
    print(f"\n{label}: {rpm:g} rpm")
    candidate_fs = 48_000 if label == "healthy" else 12_000
    orders = (1, 4, 8, 12) if label == "healthy" else (1, 2, 3)
    for order in orders:
        target = order * (rpm / 60) / candidate_fs
        nearby = peak_bins[np.abs(freq[peak_bins] - target) <= 2 * bin_width]
        if nearby.size:
            peak_bin = nearby[np.argmax(magnitude_db[nearby])]
            print(
                f"  {order}x: {freq[peak_bin]:.6f} cycles/sample "
                f"({magnitude_db[peak_bin]:.1f} dB; expected {target:.6f})"
            )
        else:
            print(f"  {order}x: no independent peak within 2 bins of {target:.6f}")

normal_freq = spectra["healthy"][0]
fault_freq = spectra["inner-race fault"][0]
normal_peaks = spectra["healthy"][2]
fault_peaks = spectra["inner-race fault"][2]
for order in (1, 2, 3):
    normal_target = order * (normal_rpm / 60) / 48_000
    fault_target = order * (fault_rpm / 60) / 12_000
    normal_near = normal_peaks[np.abs(normal_freq[normal_peaks] - normal_target) <= 2 * bin_width]
    fault_near = fault_peaks[np.abs(fault_freq[fault_peaks] - fault_target) <= 2 * bin_width]
    if normal_near.size and fault_near.size:
        normal_line = normal_freq[normal_near[0]]
        fault_line = fault_freq[fault_near[0]]
        print(
            f"{order}x comparison: 97.mat {normal_line:.6f} vs "
            f"105.mat {fault_line:.6f} cycles/sample "
            f"(ratio {normal_line / fault_line:.3f})"
        )

## Resample the normal recording to 12 kHz

The comparison supports 48 kHz for `97.mat`, so use polyphase resampling with an anti-aliasing filter to convert it to the 12 kHz rate used by `105.mat`. This keeps each sample on the same time scale for later feature and envelope comparisons.


In [ ]:
normal_48k = signals["healthy"][1]
fault_12k = signals["inner-race fault"][1]
normal_12k = resample_signal(normal_48k, source_fs=48_000, target_fs=12_000)
print(f"97.mat: {normal_48k.size:,} samples at 48 kHz")
print(f"Resampled normal: {normal_12k.size:,} samples at 12 kHz")
print(f"105.mat fault: {fault_12k.size:,} samples at 12 kHz")
print(
    f"Duration preserved: {normal_48k.size / 48_000:.3f} s -> " f"{normal_12k.size / 12_000:.3f} s"
)

## Envelope spectrum for the inner-race fault

Bearing impacts excite higher-frequency vibration that repeats at bearing defect rates. Bandpass a carrier region, take the Hilbert envelope, then inspect its low-frequency spectrum. CWRU lists the drive-end 6205 bearing inner-race frequency as 5.4152 times shaft speed, so at 1797 rpm BPFI is about 162.2 Hz. The 2.5–3.5 kHz carrier band gave the clearest BPFI-to-background contrast in this recording among the bands explored.

Source: [CWRU bearing information](https://engineering.case.edu/bearingdatacenter/bearing-information).


In [ ]:
FS = 12_000
CARRIER_BAND_HZ = (2_500, 3_500)
BPFI_ORDERS = 5.4152
SHAFT_HZ = signals["inner-race fault"][2] / 60
BPFI_HZ = BPFI_ORDERS * SHAFT_HZ
SEGMENT_SAMPLES = 60_000

normal_segment = normal_12k[:SEGMENT_SAMPLES]
fault_segment = fault_12k[:SEGMENT_SAMPLES]
normal_env = envelope(normal_segment, FS, CARRIER_BAND_HZ)
fault_env = envelope(fault_segment, FS, CARRIER_BAND_HZ)
env_freq, normal_env_spectrum = envelope_spectrum(normal_segment, FS, CARRIER_BAND_HZ)
_, fault_env_spectrum = envelope_spectrum(fault_segment, FS, CARRIER_BAND_HZ)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
time_s = np.arange(3_000) / FS
axes[0].plot(time_s, normal_env[:3_000], label="97.mat normal, resampled to 12 kHz")
axes[0].plot(time_s, fault_env[:3_000], label="105.mat inner-race fault", alpha=0.8)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Envelope amplitude")
axes[0].legend()

for label, magnitude in (("normal", normal_env_spectrum), ("inner-race fault", fault_env_spectrum)):
    axes[1].plot(env_freq[1:], 20 * np.log10(magnitude[1:]), label=label)
for harmonic in range(1, 5):
    axes[1].axvline(
        harmonic * BPFI_HZ,
        color="tab:red",
        linestyle=":" if harmonic > 1 else "--",
        alpha=0.65,
        label="predicted BPFI harmonics" if harmonic == 1 else None,
    )
axes[1].set_xlim(0, 1_000)
axes[1].set_xlabel("Envelope frequency (Hz)")
axes[1].set_ylabel("Envelope spectrum (dB re 1 recorded unit)")
axes[1].legend()
axes[1].grid(True, alpha=0.2)
plt.show()

print(f"Shaft speed: {SHAFT_HZ:.2f} Hz; predicted BPFI: {BPFI_HZ:.2f} Hz")
bpfi_mask = (env_freq >= BPFI_HZ - 2) & (env_freq <= BPFI_HZ + 2)
bpfi_bins = np.flatnonzero(bpfi_mask)
normal_bpfi_bin = bpfi_bins[np.argmax(normal_env_spectrum[bpfi_bins])]
fault_bpfi_bin = bpfi_bins[np.argmax(fault_env_spectrum[bpfi_bins])]
normal_bpfi_amp = normal_env_spectrum[normal_bpfi_bin]
fault_bpfi_amp = fault_env_spectrum[fault_bpfi_bin]
print(f"Normal BPFI-region peak: {env_freq[normal_bpfi_bin]:.1f} Hz, {normal_bpfi_amp:.4g}")
print(f"Fault BPFI-region peak: {env_freq[fault_bpfi_bin]:.1f} Hz, {fault_bpfi_amp:.4g}")
print(f"Fault/normal BPFI-band ratio: {fault_bpfi_amp / normal_bpfi_amp:.1f}x")